# Assignment: Data Wrangling

### Reading material: `tidy_data.pdf`

**Q1.** This question provides some practice cleaning variables which have common problems.
1. For `./data/airbnb_hw.csv`, clean the `Price` variable as well as you can, and explain the choices you make. How many missing values do you end up with? (Hint: What happens to the formatting when a price goes over 999 dollars, say from 675 to 1,112?)
2. For the Minnesota police use of for data, `./data/mn_police_use_of_force.csv`, clean the `subject_injury` variable, handling the NA's; this gives a value `Yes` when a person was injured by police, and `No` when no injury occurred. What proportion of the values are missing? Is this a concern? Cross-tabulate your cleaned `subject_injury` variable with the `force_type` variable. Are there any patterns regarding when the data are missing? 

**Q2.** Go to https://sharkattackfile.net/ and download their dataset on shark attacks (Hint: `GSAF5.xls`).

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [To be provided]

**Date:** [To be provided]

### Question 1 Part 1

In [ ]:
!pip install pandas
!pip install matplotlib

In [ ]:
import pandas as pd
data = pd.read_csv("data/airbnb_hw.csv")
print("First 20 values of the 'Price' column:")
print(data["Price"].head(20))

print("The Data Type of the 'Price' column:")
print(data["Price"].dtype)

print(f"Missing values: {data['Price'].isna().sum()}")

print("Value counts of the 'Price' column (top 20):")
data["Price"].value_counts().tail(20)

Based on my investigation, I realized that the 'Price' column is the str datatype. For prices that are above 999, the value contains a column. So my solution would involved removing the comma with a an empty string. Then I can cleanly convert the 'Price' column from the str datatype to the int datatype.

In [ ]:
# Replace commas in the 'Price' column with an empty string
data["Price"] = data["Price"].str.replace(',', '')

# Convert the 'Price' column to int
data["Price"] = pd.to_numeric(data["Price"])

#Results
print(f"Data type: {data['Price'].dtype}")
print(f"Missing values: {data['Price'].isna().sum()}")
print(f"\nFirst 20 values:")
print(data["Price"].head(20))
print(data["Price"].describe())

### Question 1 Part 2

In [ ]:
data2 = pd.read_csv("data/mn_police_use_of_force.csv")
print(f"Missing values in 'subject_injury': {data2['subject_injury'].isna().sum()}")
print(f"Proportion of missing values in 'subject_injury': {(data2['subject_injury'].isna().sum() / len(data2))*100}%")
print(data2['subject_injury'].value_counts())

data2[['subject_injury', 'force_type']].head()

Based on my initial look at the 'subject_injury' and the 'force_type' column, I realized that the the 'subject_injury column has many missing values. Approxiametely 76 percent of this column are missing values. Since the goal is to find the meaning behind this missing values, cross tabulating does not work at this moment because the missing value is not a unique value that is counted. So the solution I would use is to fill the missing value. 

In [ ]:
#fill missing values in the 'subject_injury' column with 'Missing'
data2['subject_injury'] = data2['subject_injury'].fillna('Missing')
print(data2[['subject_injury', 'force_type']].head())

#Perform the crosstab analysis
crosstab = pd.crosstab(data2['subject_injury'], data2['force_type'])
print(crosstab)

crosstab_analysis = pd.DataFrame(
    index=crosstab.columns,
    columns=['Frequency_Missing']
)
crosstab_analysis['Frequency_Missing'] = (crosstab.loc['Missing'].values)
print(crosstab_analysis)


After filling the missing values with the string 'Missing', I am able to see the relationship between the missing values in 'subject_injury" and force type using the cross tab. I am noticing that bodily force, chemical irritant, and maximal restraint technique of most, if not all, values as missing, which is an interesting observation. 

### Question 2
Go to https://sharkattackfile.net/ and download their dataset on shark attacks (Hint: `GSAF5.xls`).

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?

In [ ]:
#Load the data
shark_df = pd.read_csv("data/GSAF5.csv", encoding="latin-1")

#Remove Empty Columns
shark_df = shark_df.dropna(axis=1, how='all')
shark_df = shark_df.drop(columns = ['Unnamed: 21', 'Unnamed: 22'])

#Display basic information about the dataframe
print(shark_df.head())
print(shark_df.info())
print(shark_df.describe())
print(shark_df.columns)

In [ ]:
shark_df['Year'].describe()

After cleaning the year column, the range of values that I see are from 0 to 2026. I doubt this data has been tracking shark attacks since the beginning of time.

In [ ]:
shark_df_filter = shark_df.loc[shark_df['Year'] >= 1940]
print(shark_df_filter.head())
plt.hist(shark_df_filter['Year'], bins=20, edgecolor='black')
plt.title("Distribution of Year")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.show()

Based on the graph that I plotted for the number of shark attacks over time, it seems there was a small peak in of shark attacks in the 1960s then a brief drop afterwards. Then the number of attacks have increased over time and peaked in the 2010s. 

In [ ]:
shark_df_filter["Age"].describe()
shark_df_filter["Age_Clean"] = pd.to_numeric(shark_df_filter["Age"], errors='coerce')
shark_df_filter["Age_Clean"].fillna(shark_df_filter["Age_Clean"].median(), inplace=True)
shark_df_filter["Age_Clean"].describe()

In [ ]:
import matplotlib.pyplot as plt
plt.hist(shark_df_filter["Age_Clean"], bins=5, edgecolor='black')
plt.title("Distribution of Age_Clean")
plt.xlabel("Age_Clean")
plt.ylabel("Frequency")

In [ ]:
victim_gender = shark_df_filter["Sex"].value_counts().loc[["M", "F"]]
print(f"The proportion of male victims: {victim_gender['M'] / victim_gender.sum() * 100:.2f}%")
print(f"The proportion of female victims: {victim_gender['F'] / victim_gender.sum() * 100:.2f}%")


The proportion of male victims is 85.73 percent, while the proportion of female victims is 14.27 percent.

In [ ]:
shark_df_filter['Type'].value_counts()
shark_df_filter['Type'] = shark_df_filter['Type'].str.lower()
shark_df_filter['Type'] = shark_df_filter['Type'].replace(['watercraft', 'boat', 'sea disaster'], 'unprovoked')
shark_df_filter['Type'] = shark_df_filter['Type'].replace(['invalid', 'questionable', 'unconfirmed', 'unverified', "?", 'under investigation'], 'unknown')
shark_df_filter['Type'] = shark_df_filter['Type'].replace(' provoked', 'provoked')
shark_df_filter['Type'].value_counts()

In [ ]:
shark_type_freq = shark_df_filter['Type'].value_counts()
print(shark_type_freq)
print(f"The proportion of unprovoked attacks: {shark_type_freq['unprovoked'] / shark_type_freq.sum() * 100:.2f}%")

The proportion of unprovoked attacks is 82.74 percent.


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]